# Başlanğıc RAG (Retrieval-Augmented Generation) — OpenRouter ilə

Bu versiyada retrieval (embedding + FAISS) birinci notebookla eynidir — hələ də lokal işləyir və pulsuzdur. Fərq generasiya addımındadır: cavabı lokal kiçik modeldə yox, [OpenRouter](https://openrouter.ai) API vasitəsilə istənilən böyük modeldə (GPT, Claude, Llama, Gemini və s.) generasiya edirik.

## Retrieval (axtarış) nədir?

RAG-ın "R" hərfi buradan gəlir (Retrieval-Augmented Generation). Model sualı cavablandırmazdan əvvəl, əlaqəli sənədləri böyük bir kolleksiyadan tapır ("retrieve" edir) və bunları modelə əlavə kontekst kimi verir.

Bu niyə lazımdır: dil modelləri öz təlim datasında olmayan və ya köhnəlmiş məlumatı bilmir. Retrieval bu boşluğu doldurur — modelin cavabını sənin öz sənədlərinə (PDF, daxili sənədlər, wiki və s.) əsaslandırmasına imkan verir və uydurma (hallucination) ehtimalını azaldır.


## Embedding nədir?

Embedding — mətni (söz, cümlə, paraqraf) ədədi vektora çevirmə prosesidir. Mənaca yaxın mətnlərin vektorları da bir-birinə yaxın olur. Məsələn "pişik" və "pişik balası" vektorları yaxın olacaq, "pişik" və "avtomobil" isə uzaq.

RAG-da embedding belə istifadə olunur:
1. Bütün sənədlər əvvəlcədən vektora çevrilir və saxlanılır (vektor bazası / indeks).
2. İstifadəçi sual verəndə, sualın özü də eyni modeldə vektora çevrilir.
3. Sual vektoruna ən yaxın sənəd vektorları tapılır (məsafə ölçüsü ilə, məs. cosine ya L2) — bunlar ən əlaqəli sənədlərdir.

Hər iki notebookda embedding üçün `sentence-transformers` kitabxanasından `all-MiniLM-L6-v2` modeli istifadə olunur — kiçik (~80MB), sürətli və CPU-da belə rahat işləyir.


## OpenRouter API açarı

1. [openrouter.ai](https://openrouter.ai) saytında qeydiyyatdan keç
2. Dashboard → Keys bölməsindən API key yarat
3. Model seçimi: [openrouter.ai/models](https://openrouter.ai/models) səhifəsində modelin adının sonunda `:free` olanlar pulsuzdur, digərləri balansdan kredit çıxarır (adətən çox ucuzdur).

## Bu notebookda addımlar

1. Kitabxanaların qurulması
2. API açarının daxil edilməsi
3. Nümunə sənədlər (kiçik korpus)
4. Sənədləri embedding-ə çevirmək (lokal)
5. FAISS ilə vektor indeksi qurmaq (lokal)
6. Retrieval funksiyası (lokal)
7. RAG pipeline: retrieval (lokal) + generasiya (OpenRouter API)
8. Test sualları

In [ ]:
!pip install -q sentence-transformers faiss-cpu openai

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from openai import OpenAI
import getpass

### API açarını daxil et
Açar Colab çıxışında görünməsin deyə `getpass` istifadə olunur.

In [ ]:
OPENROUTER_API_KEY = getpass.getpass("OpenRouter API key daxil et: ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

### 1. Nümunə sənədlər
Realda bunlar PDF, daxili sənədlər, dataset sətirləri və s. ola bilər. Sadəlik üçün bir neçə qısa mətn paraqrafı istifadə edirik.

In [ ]:
documents = [
    "The Eiffel Tower is a wrought-iron lattice tower located in Paris, France. It was completed in 1889 and stands about 330 meters tall.",
    "Photosynthesis is the process by which green plants use sunlight to synthesize food from carbon dioxide and water.",
    "Python is a high-level, interpreted programming language known for its readability and wide use in data science and AI.",
    "The Great Wall of China is a series of fortifications built to protect Chinese states from invasions, stretching thousands of kilometers.",
    "Machine learning is a subset of artificial intelligence where systems learn patterns from data rather than being explicitly programmed.",
    "The human heart pumps blood through the circulatory system, delivering oxygen and nutrients to tissues throughout the body.",
]
print(f"{len(documents)} sənəd yükləndi")


### 2. Embedding modelini yükləmək və sənədləri kodlaşdırmaq
Bu addım hələ də tamamilə lokaldır, OpenRouter-ə aid deyil.

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True)
print("Embedding ölçüsü:", doc_embeddings.shape)  # (sənəd sayı, vektor ölçüsü)


### 3. FAISS vektor indeksi

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)
print(f"İndeksdə {index.ntotal} sənəd var, hər biri {dimension} ölçülü vektor")


### 4. Retrieval funksiyası

In [ ]:
def retrieve(query, k=2):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [documents[i] for i in indices[0]]

# sürətli yoxlama
retrieve("Which language is popular in AI?")


### 5. Generasiya — OpenRouter API çağırışı

`MODEL_NAME` dəyişənini istənilən modellə əvəz edə bilərsən, məsələn `openai/gpt-4o-mini`, `anthropic/claude-3.5-haiku`, `google/gemini-2.0-flash-001`. Aşağıdakı `:free` modeli sınaq üçündür — hazırda mövcud pulsuz modelləri openrouter.ai/models səhifəsindən yoxla, çünki siyahı vaxtaşırı dəyişir.

In [ ]:
MODEL_NAME = "meta-llama/llama-3.1-8b-instruct:free"  # istəsən başqa modellə əvəz et

### 6. RAG pipeline — retrieval (lokal) + generasiya (OpenRouter)

In [ ]:
def rag_answer(query, k=2):
    context_docs = retrieve(query, k=k)
    context = "\n".join(context_docs)
    prompt = f"Aşağıdakı kontekstə əsaslanaraq suala qısa cavab ver.\n\nContext:\n{context}\n\nSual: {query}"

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
    )
    return completion.choices[0].message.content, context_docs

answer, sources = rag_answer("What is Python used for?")
print("Cavab:", answer)
print("\nİstifadə olunan mənbələr:")
for s in sources:
    print("-", s)

### 7. Bir neçə test sualı

In [ ]:
for q in [
    "How tall is the Eiffel Tower?",
    "What does the heart do?",
    "What is machine learning?",
]:
    ans, _ = rag_answer(q)
    print(f"Sual: {q}\nCavab: {ans}\n")

## Növbəti addımlar

- Öz sənədlərini (PDF, .txt, wiki export) yükləyib `documents` siyahısını onlarla əvəz et
- Uzun sənədləri kiçik hissələrə bölmək (chunking) strategiyalarını araşdır — bir paraqraf çox uzun olanda embedding keyfiyyəti aşağı düşür
- `IndexFlatL2` əvəzinə böyük data üçün `IndexIVFFlat` kimi sürətli FAISS indekslərinə bax
- FAISS əvəzinə hazır vektor bazalarını (Chroma, Qdrant, Pinecone) sınamaq
- Retrieval keyfiyyətini ölçmək üçün "hansı sənəd hansı suala aid olmalıdır" siyahısı hazırlayıb dəqiqliyi hesablamaq

- Fərqli `MODEL_NAME` dəyərləri ilə eyni sualları sınayıb cavab keyfiyyətini müqayisə et
- OpenRouter dashboard-da istifadə/xərc izləməsinə bax